# Notebook 2: ResNet50 + U-Net Segmentation
Trains U-Net only on Tp images with Mask as ground truth.

In [6]:
import tensorflow as tf
import os
import numpy as np
from tensorflow.keras import layers, models
from PIL import Image


In [2]:
IMG_SIZE = 224
BATCH_SIZE = 8

In [3]:
def load_image_pil(path):
    """Load TIFF image using PIL and convert to numpy array"""
    path = path.numpy().decode('utf-8')
    img = Image.open(path).convert('RGB')
    img = img.resize((IMG_SIZE, IMG_SIZE))
    return np.array(img, dtype=np.float32) / 255.0

def load_image(path):
    img = tf.py_function(load_image_pil, [path], tf.float32)
    img.set_shape((IMG_SIZE, IMG_SIZE, 3))
    return img

def load_mask(path):
    mask = tf.io.read_file(path)
    mask = tf.image.decode_png(mask, channels=1)
    mask = tf.image.resize(mask, (IMG_SIZE, IMG_SIZE))
    return tf.cast(mask > 0, tf.float32)


In [8]:
def build_dataset(base_dir):
    tp_dir = os.path.join(base_dir, 'Tp')
    mask_dir = os.path.join(base_dir, 'Mask')

    imgs = sorted(os.listdir(tp_dir))
    img_paths = [os.path.join(tp_dir, f) for f in imgs]
    # Convert Tp filename to mask filename: remove .tif, add _gt.png
    mask_paths = [os.path.join(mask_dir, os.path.splitext(f)[0] + '_gt.png') for f in imgs]

    ds = tf.data.Dataset.from_tensor_slices((img_paths, mask_paths))
    ds = ds.map(lambda x, y: (load_image(x), load_mask(y)),
                num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = build_dataset('../split_dataset/train')
val_ds = build_dataset('../split_dataset/val')


In [32]:
encoder = tf.keras.applications.ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

skip_layers = [
    'conv1_relu',
    'conv2_block3_out',
    'conv3_block4_out',
    'conv4_block6_out'
]
skips = [encoder.get_layer(n).output for n in skip_layers]


In [33]:
def decoder_block(x, skip, filters):
    x = layers.UpSampling2D((2,2))(x)
    x = layers.Concatenate()([x, skip])
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    return x


In [34]:
x = encoder.output
x = decoder_block(x, skips[3], 512)
x = decoder_block(x, skips[2], 256)
x = decoder_block(x, skips[1], 128)
x = decoder_block(x, skips[0], 64)
x = layers.UpSampling2D((2,2))(x)
out = layers.Conv2D(1, 1, activation='sigmoid')(x)

unet = models.Model(encoder.input, out)


In [35]:
def dice_loss(y_true, y_pred, smooth=1):
    intersection = tf.reduce_sum(y_true * y_pred)
    return 1 - (2*intersection + smooth) / (
        tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth
    )

unet.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=lambda y_true, y_pred:
        tf.keras.losses.BinaryCrossentropy()(y_true, y_pred) + dice_loss(y_true, y_pred),
    metrics=['accuracy']
)


In [ ]:
unet.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)
unet.save('resnet50_unet_tamper.keras')


Epoch 1/10
394/394 ━━━━━━━━━━━━━━━━━━━━ 10267s 26s/step - accuracy: 0.9051 - loss: 1.1453 - val_accuracy: 0.9029 - val_loss: 1.4764
Epoch 2/10
394/394 ━━━━━━━━━━━━━━━━━━━━ 3383s 9s/step - accuracy: 0.9048 - loss: 1.0808 - val_accuracy: 0.8939 - val_loss: 1.4828
Epoch 3/10
394/394 ━━━━━━━━━━━━━━━━━━━━ 1864s 5s/step - accuracy: 0.9185 - loss: 1.0103 - val_accuracy: 0.8986 - val_loss: 1.3725
Epoch 4/10
 99/394 ━━━━━━━━━━━━━━━━━━━━ 22:48 5s/step - accuracy: 0.8680 - loss: 0.9947